# Benchmarks

## Initialize

In [1]:
#library(Rmisc)
library(dtplyr)
library(tidyverse)
library(glue)
library(arrow)
library(patchwork)
library(data.table)
library("jsonlite")
library(ggthemes)

Warning message:
“package ‘dtplyr’ was built under R version 4.0.5”
Warning message:
“package ‘tidyverse’ was built under R version 4.0.5”
── Attaching packages ─────────────────────────────────────── tidyverse 1.3.2 ──
✔ ggplot2 3.3.6      ✔ purrr   0.3.5 
✔ tibble  3.1.8      ✔ dplyr   1.0.10
✔ tidyr   1.2.1      ✔ stringr 1.5.0 
✔ readr   2.1.2      ✔ forcats 0.5.2 
Warning message:
“package ‘ggplot2’ was built under R version 4.0.5”
Warning message:
“package ‘tibble’ was built under R version 4.0.5”
Warning message:
“package ‘tidyr’ was built under R version 4.0.5”
Warning message:
“package ‘readr’ was built under R version 4.0.5”
Warning message:
“package ‘dplyr’ was built under R version 4.0.5”
Warning message:
“package ‘forcats’ was built under R version 4.0.5”
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
Warning message:
“package ‘glue’ was built under R version 4.0.

In [2]:
if (grepl("sc", Sys.info()[["nodename"]], fixed=TRUE)) {
    base_path = "/sc-projects/sc-proj-ukb-cvd"
} else {
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"}
print(base_path)

project_label="22_medical_records"
project_path = glue("{base_path}/results/projects/{project_label}")
figure_path = glue("{project_path}/figures")
output_path = glue("{project_path}/data")

experiment = 230425
experiment_path = glue("{output_path}/{experiment}")

[1] "/sc-projects/sc-proj-ukb-cvd"


In [3]:
base_size = 8
title_size = 10
facet_size = 9
geom_text_size=3
theme_set(theme_classic(base_size = base_size) + 
          theme(strip.background = element_blank(), plot.title=element_text(size=title_size, hjust=0), 
                strip.text.x = element_text(size = facet_size),axis.title=element_text(size=10), axis.text=element_text(size=8, color="black"),
                legend.position="bottom", axis.line = element_line(size = 0.2), axis.ticks=element_line(size=0.2), panel.grid.major.y=element_line()))

In [4]:
colors_dict = read_json("colors.json")
color_map <- c(
    "Identity(AgeSex)+MLP" = colors_dict$pastel$red$mid,
    "Identity(Records)+MLP" = colors_dict$pastel$red$mid,
    "GNN(Records)+MLP" = colors_dict$pastel$red$mid,
    "Identity(AgeSex+Records)+MLP" = colors_dict$pastel$red$mid,
    "GNN(AgeSex+Records)+MLP" = colors_dict$pastel$red$mid
)

In [5]:
endpoint_defs = arrow::read_feather(glue("{output_path}/phecode_defs_220306.feather")) %>% arrange(endpoint)
endpoints_md = fread(glue("{experiment_path}/endpoints.csv"), colClasses=c("phecode"="character"))
endpoints = sort(endpoints_md$endpoint)

In [6]:
endpoint_map = endpoint_defs$phecode_string
names(endpoint_map) =  endpoint_defs$endpoint
endpoint_order = (endpoint_defs %>% arrange(as.numeric(phecode)))$endpoint

In [7]:
endpoint_selection = c(
    
   'phecode_401', #  "Hypertension", # intervention
    'phecode_202', #  "Diabetes mellitus", # intervention
    'phecode_416-21', #  "Atrial fibrillation", # intervention
    'phecode_468', #  "Pneumonia", # intervention
    'phecode_474', #  "Chronic obstructive pulmonary disease [COPD]", # interventio
    'phecode_583', #  "Chronic kidney disease", # intervention
    
    'phecode_404', #  "Ischemic heart disease",
    'phecode_404-1', #  "Myocardial infarction [Heart attack]", # intervention
    'phecode_431-11', #  "Cerebral infarction [Ischemic stroke]",
    'phecode_424', #  "Heart failure", # intervention
    'phecode_420', #  "Cardiac arrest", # intervention
    'OMOP_4306655', #  "All-Cause Death", # intervention
    
    'phecode_438-11',   #  "Abdominal aortic aneurysm",
    'phecode_440-3',#  "Pulmonary embolism", # intervention
    'phecode_413-21',#  "Aortic stenosis", # intervention
    'phecode_413-11', #  "Mitral valve insufficiency",
    'phecode_410-2',#  "Endocarditis",
    'phecode_400',#  "Rheumatic fever and chronic rheumatic heart diseases",	
    
    'phecode_164', #  "Anemia", # intervention
    'phecode_718',  #  "Back pain", # intervention
    'phecode_324-11', #  "Parkinson's disease (Primary)",
    'phecode_705-1', #  "Rheumatoid arthritis", # NEW + interventio
    'phecode_665', #  "Psoriasis", # interesting
    'phecode_284'#  "Suicide ideation and attempt or self harm" # intervention
)
endpoint_defs = endpoint_defs %>% 
    mutate(name = phecode_string) %>%
    mutate(name = 
           case_when( 
               phecode_string == "Myocardial infarction [Heart attack]"~"Myocardial infarction",
               phecode_string == "Cerebral infarction [Ischemic stroke]"~"Ischemic stroke",
               phecode_string == "Chronic obstructive pulmonary disease [COPD]"~"COPD",
               phecode_string == "Mitral valve insufficiency"~"Mitral insufficiency",
               phecode_string == "Parkinson's disease (Primary)"~"Parkinson's",
               phecode_string == "Suicide ideation and attempt or self harm"~"Suicide attempt",
               phecode_string == "Ischemic heart disease"~"Ischemic HD",
               phecode_string == "Chronic kidney disease"~"Chronic KD",
               phecode_string == "Rheumatic fever and chronic rheumatic heart diseases"~"Rheumatic HD",
               phecode_string == "Abdominal aortic aneurysm"~"Abdominal AA",
                  TRUE ~ name)
           )
            
endpoint_map = endpoint_defs$name
names(endpoint_map) =  endpoint_defs$endpoint
#endpoint_order = (endpoint_defs %>% arrange(as.numeric(phecode)))$endpoint
endpoint_order = endpoint_selection

## Load data

# Load Benchmarks

In [9]:
list.files("/sc-projects/sc-proj-ukb-cvd/results/projects/22_medical_records/data/220823_allofus/230502_revision")

[1] "230502_result_ukbbparams_ubr_cardiac_arrest_clean_cindices_revision.feather"                   
 [2] "230502_result_ukbbparams_ubr_cardiac_arrest_delta_clean_cindices.feather"                      
 [3] "230502_result_ukbbparams_ubr_cindices.csv"                                                     
 [4] "230502_result_ukbbparams_ubr_clean_cindices.feather"                                           
 [5] "230503_bootstrap_results_revision_ensemble_128.feather"                                        
 [6] "230503_bootstrap_results_revision_ensemble.feather"                                            
 [7] "230503_result_ukbbparams_ubr_cardiac_arrest_clean_cindices_revision_ensemble.feather"          
 [8] "230503_result_ukbbparams_ubr_cardiac_arrest_delta_clean_cindices_revision_ensemble (1).feather"
 [9] "230503_result_ukbbparams_ubr_cardiac_arrest_delta_clean_cindices_revision_ensemble.feather"    
[10] "230503_result_ukbbparams_ubr_cindices_ensemble.csv"                                            
[11] "230503_result_ukbbparams_ubr_clean_cindices_ensemble.feather"                                  
[12] "bootstrap_results.feather"                                                                     
[13] "endpoints.csv"                                                                                 
[14] "phecode_defs_220306.feather"

In [8]:
benchmark_endpoints_aou_raw = arrow::read_feather(
    "/sc-projects/sc-proj-ukb-cvd/results/projects/22_medical_records/data/220823_allofus/230502_revision/230503_bootstrap_results_revision_ensemble.feather") %>%
    mutate(endpoint=str_replace_all(endpoint, "\\.", "\\-")) %>%
    left_join(endpoints_md) 

Joining, by = "endpoint"


In [9]:
benchmark_endpoints_aou_raw

score,cindex,endpoint,num_events,num_included,partition,age_coeff,uuid,is_male_coeff,V1,eligable,n,freq,phecode,phecode_string,phecode_category,sex,ICD10_only,phecode_top,leaf
<chr>,<dbl>,<chr>,<int>,<int>,<chr>,<dbl>,<chr>,<dbl>,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>
AgeSex+MedicalHistory_UKBBParams,0.7157854,phecode_215,514,228753,ensemble,NA,335206507169795216104456980405471543300,NA,375,228738,641,0.0028023328,215,Testicular dysfunction,Endo,Male,0,215,0
AgeSex_AoUCPH,0.5206259,phecode_215,490,80551,ensemble,0.09331761,335206507169795216104456980405471543300,NA,375,228738,641,0.0028023328,215,Testicular dysfunction,Endo,Male,0,215,0
AgeSex+MedicalHistory_UKBBParams,0.6002514,phecode_460-1,13663,171117,ensemble,NA,80540469189667082795357966486784311300,NA,1055,376051,74826,0.1989783301,460.1,Acute upper respiratory infection,Resp,Both,0,460,1
AgeSex_AoUCPH,0.5780146,phecode_460-1,13663,171117,ensemble,-0.22865554,80540469189667082795357966486784311300,-0.257403445,1055,376051,74826,0.1989783301,460.1,Acute upper respiratory infection,Resp,Both,0,460,1
AgeSex+MedicalHistory_UKBBParams,0.7061874,phecode_381,810,229180,ensemble,NA,80226635346005329756943672994815934468,NA,819,498337,2681,0.0053798935,381,Strabismus,Eye,Both,0,381,0
AgeSex_AoUCPH,0.6354237,phecode_381,810,229180,ensemble,0.47848535,80226635346005329756943672994815934468,0.235420245,819,498337,2681,0.0053798935,381,Strabismus,Eye,Both,0,381,0
AgeSex+MedicalHistory_UKBBParams,0.6907239,phecode_530-1,485,229883,ensemble,NA,80435233590525886045382446400723353604,NA,1277,492663,5834,0.0118417661,530.1,Anal fissure,GI,Both,0,530,1
AgeSex_AoUCPH,0.6124018,phecode_530-1,485,229883,ensemble,-0.37257971,80435233590525886045382446400723353604,-0.088041957,1277,492663,5834,0.0118417661,530.1,Anal fissure,GI,Both,0,530,1
AgeSex+MedicalHistory_UKBBParams,0.8161655,phecode_323-1,92,231798,ensemble,NA,80168163377506552388899471122083479556,NA,516,502246,379,0.0007546103,323.1,Hereditary ataxia,Neuro,Both,0,323,0


In [10]:
benchmark_endpoints_aou =  benchmark_endpoints_aou_raw %>% select(score, cindex, endpoint, num_included, num_events, partition) %>% rename(iteration=partition) %>% filter(num_events>=100) %>%
    filter(score %in% c("AgeSex_AoUCPH", "AgeSex+MedicalHistory_AoUCPH")) %>% mutate(cohort="AoU") %>% mutate(score=str_remove_all(score, "_AoUCPH")) %>% 
    mutate(score=str_replace_all(score, "AgeSex", "Age+Sex")) #%>% pivot_wider(names_from="score", values_from="cindex") %>% arrange(partition)

In [11]:
name = "benchmarks_cindex_220627"
benchmark_endpoints = arrow::read_feather(glue("{experiment_path}/{name}.feather")) %>% left_join(endpoints_md) %>% mutate(cohort="UKB")

ERROR: Error: IOError: Failed to open local file '/sc-projects/sc-proj-ukb-cvd/results/projects/22_medical_records/data/230425/benchmarks_cindex_220627.feather'. Detail: [errno 2] No such file or directory


In [15]:
benchmark_endpoint_all = bind_rows(benchmark_endpoints, benchmark_endpoints_aou) %>% filter(score %in% c("Age+Sex", "Age+Sex+MedicalHistory")) %>% select(endpoint, score, cindex, cohort, iteration) %>% arrange(endpoint) %>%
    pivot_wider(names_from="score", values_from="cindex") %>% unnest(cols=c("Age+Sex", "Age+Sex+MedicalHistory")) %>% arrange(endpoint) %>% mutate(delta=`Age+Sex+MedicalHistory` - `Age+Sex`) 

ERROR: Error in list2(...): object 'benchmark_endpoints' not found


In [13]:
library(ggdist)
benchmark_endpoint_agg = benchmark_endpoint_all %>% group_by(endpoint, cohort) %>% median_qi(`Age+Sex`, `Age+Sex+MedicalHistory`, `delta`) %>% arrange(endpoint, cohort)

Warning message:
“package ‘ggdist’ was built under R version 4.0.5”


ERROR: Error in group_by(., endpoint, cohort): object 'benchmark_endpoint_all' not found


In [14]:
benchmark_endpoint_agg_md = benchmark_endpoint_agg %>% left_join(endpoints_md) %>% select(cohort, endpoint, `Age+Sex`, `Age+Sex+MedicalHistory`, delta, freq, phecode_string, phecode_category) %>% ungroup()

ERROR: Error in left_join(., endpoints_md): object 'benchmark_endpoint_agg' not found


In [ ]:
benchmark_endpoint_agg_md

In [ ]:
plot_width = 5; plot_height=5; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
endpoint_order = (benchmark_endpoint_agg_md %>% filter(cohort=="UKB") %>% arrange(delta))$endpoint
ggplot(benchmark_endpoint_agg_md %>% filter(endpoint %in% endpoint_order) %>% filter(!is.na(delta)), aes(x=as.numeric(factor(endpoint, levels=endpoint_order)), y=delta, alpha=freq, color=cohort)) + geom_point() + geom_smooth()

In [ ]:
benchmark_endpoints_aou_raw

In [ ]:
benchmark_endpoints_aou =  benchmark_endpoints_aou_raw %>% select(score, cindex, endpoint, num_included, num_events, partition) %>% rename(iteration=partition) %>% filter(num_events>=100) %>%
    filter(score %in% c("AgeSex_AoUCPH", "AgeSex+MedicalHistory_UKBBParams")) %>% mutate(cohort="AoU") %>% mutate(score=str_remove_all(score, "_AoUCPH|_UKBBParams")) %>% 
    mutate(score=str_replace_all(score, "AgeSex", "Age+Sex")) #%>% pivot_wider(names_from="score", values_from="cindex") %>% arrange(partition)

In [ ]:
name = "benchmarks_cindex_220627"
benchmark_endpoints = arrow::read_feather(glue("{experiment_path}/{name}.feather")) %>% left_join(endpoints_md) %>% mutate(cohort="UKB")

In [ ]:
benchmark_endpoint_all = bind_rows(benchmark_endpoints, benchmark_endpoints_aou) %>% filter(score %in% c("Age+Sex", "Age+Sex+MedicalHistory")) %>% select(endpoint, score, cindex, cohort, iteration) %>% arrange(endpoint) %>%
    pivot_wider(names_from="score", values_from="cindex") %>% unnest(cols=c("Age+Sex", "Age+Sex+MedicalHistory")) %>% arrange(endpoint) %>% mutate(delta=`Age+Sex+MedicalHistory` - `Age+Sex`) 

In [ ]:
library(ggdist)
benchmark_endpoint_agg = benchmark_endpoint_all %>% group_by(endpoint, cohort) %>% median_qi(`Age+Sex`, `Age+Sex+MedicalHistory`, `delta`) %>% arrange(endpoint, cohort)

In [ ]:
benchmark_endpoint_agg_md = benchmark_endpoint_agg %>% left_join(endpoints_md) %>% select(cohort, endpoint, `Age+Sex`, `Age+Sex+MedicalHistory`, delta, freq, phecode_string, phecode_category) %>% ungroup()

In [ ]:
benchmark_endpoint_agg_md

In [ ]:
plot_width = 5; plot_height=5; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
endpoint_order = (benchmark_endpoint_agg_md %>% filter(cohort=="UKB") %>% arrange(delta))$endpoint
ggplot(benchmark_endpoint_agg_md %>% filter(endpoint %in% endpoint_order) %>% filter(!is.na(delta)), aes(x=as.numeric(factor(endpoint, levels=endpoint_order)), y=delta, alpha=freq, color=cohort)) + geom_point() + geom_smooth()

In [ ]:
ggplot(benchmark_endpoint_agg_md %>% filter(endpoint %in% endpoint_order) %>% filter(!is.na(delta)), aes(x=freq, y=delta, alpha=freq, color=cohort)) + geom_point() + geom_smooth()

In [ ]:
benchmark_endpoint_agg_md %>% filter(endpoint %in% endpoint_order) %>% filter(!is.na(delta))

In [ ]:
benchmark_endpoint_agg_md_wide = benchmark_endpoint_agg_md %>% pivot_wider(names_from="cohort", values_from=c("Age+Sex", "Age+Sex+MedicalHistory", delta))

In [ ]:
plot_width = 5; plot_height=5; plot_res = 320
library(ggpubr)
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)
ggplot(benchmark_endpoint_agg_md_wide, aes(x=delta_UKB, y=delta_AoU)) + geom_point(alpha=0.3, size=0.5) + geom_smooth(method="lm") + stat_cor(method = "pearson", label.x = -0.3, label.y = 0.24)

In [ ]:
temp_ordered = benchmark_endpoint_agg_md_wide %>% mutate(delta=delta_AoU-delta_UKB) %>% arrange(delta) %>% filter(!is.na(delta))

In [ ]:
temp_ordered %>% group_by(phecode_category) %>% median_qi(delta)

In [ ]:
ggplot(temp_ordered, aes(x=freq, y=delta, alpha=freq)) + geom_point()

In [ ]:
ggplot(temp_ordered, aes(x=fct_reorder(endpoint, delta), y=delta, alpha=freq)) + geom_point()

In [ ]:
temp_ordered %>% filter(freq > 0.01, delta < -0.1)# %>% sample_n(50)

In [ ]:
temp_ordered %>% filter(freq > 0.01, abs(delta) < 0.01) %>% sample_n(50)

In [ ]:
benchmark_endpoints_agg = benchmark_endpoint_all %>% group_by(cohort, endpoint) %>% summarise(cindex=mean(cindex)) %>% group_by(score) %>% summarise(mean(cindex)) %>% arrange(`mean(cindex)`)

In [ ]:
endpoints_sorted = (benchmark_endpoints %>% 
    filter(score == "Age+Sex+MedicalHistory") %>% 
    group_by(endpoint, score) %>% 
    summarise(cindex=mean(cindex, na.rm=TRUE)) %>% 
    arrange(cindex) %>% ungroup())$endpoint

In [ ]:
categories_sorted = (endpoint_defs %>% distinct(phecode_category))$phecode_category

In [ ]:
benchmark_endpoints

benchmark_endpoints## General Performance

In [ ]:
plot_width = 8.25; plot_height=2.5; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)

library(ggtext)
library(ggdist)

scores_plot = c("Age+Sex", "Age+Sex+MedicalHistory")#, "AgeSexMedicalHistory")

temp = benchmark_endpoints %>% 
    filter(score %in% scores_plot) %>% 
    mutate(score = factor(score, levels=scores_plot)) %>%
    mutate(endpoint = factor(endpoint, levels=endpoints_sorted)) %>%
    ungroup() %>%
    pivot_wider(names_from=score, values_from=cindex) %>% 
    mutate(id = row_number()) %>%
    mutate(delta = `Age+Sex+MedicalHistory`-`Age+Sex`) %>%
    group_by(endpoint, phecode_string, freq, phecode_category) %>%
    median_qi(delta) %>%
    mutate(pos = case_when(delta>=0 ~ "pos", delta < 0 ~"neg")) %>%
    mutate(endpoint = fct_reorder(endpoint, delta)) %>%
    mutate(highlight = case_when(endpoint %in% endpoint_selection ~ "YES", TRUE ~ "NO")) %>%# %>% filter(endpoint %in% endpoint_sample)
    mutate(phecode_category = factor(phecode_category, levels=categories_sorted))

endpoint_order = (temp %>% arrange(delta))$endpoint

temp = temp %>% mutate(endpoint = factor(endpoint, levels=endpoint_order)) %>% ungroup() %>% 
    arrange(endpoint) %>% group_by(phecode_category) %>% mutate(endpoint = row_number()) %>%
    filter(!phecode_category %in% c("Signs/Symptoms", "Preg", "Rx", "Stat"))

overview = ggplot(temp) +
    geom_ribbon(aes(x=endpoint, ymin=0, ymax=delta), fill="black", alpha=0.2)+
    geom_point(aes(x=endpoint, y=delta, color=highlight, size=highlight, alpha=highlight)) +
    #geom_text(data=temp %>% filter(highlight=="YES"), aes(x=endpoint, y=delta+0.045, label="↓"), color="black", size=5, alpha=0.7) +
    #geom_segment(aes(x=endpoint, xend=endpoint, y=0, yend=delta, color=highlight, size=highlight), alpha=0.5)+#+
    labs(x="Endpoints", y="Delta C-Index")+
    scale_color_manual(values=c("NO"="black", "YES"="firebrick"))+
    scale_alpha_manual(values=c("NO"=0.1, "YES"=1))+
    scale_size_manual(values=c("NO"=0.01, "YES"=1))+
    #scale_colour_manual(values = c("pos"="forestgreen", "neg" = "firebrick")) + 
    #coord_polar() +
    coord_cartesian(ylim=c(-0.4, 0.4), clip = "off")+
    scale_y_continuous(expand=c(0, 0))+
    scale_x_discrete(expand=expansion(add=20))+
    facet_grid(~phecode_category, scales="free_x", space="free_x")+#, switch=TRUE)+
    #facet_grid2(~phecode_category, scales = "free", independent = "all") + 
    theme(axis.title.x=element_blank(),
        axis.text.x=element_blank(),
        axis.ticks.x=element_blank(),
        panel.grid.major=element_blank(), 
         strip.text = element_text(angle=270, hjust=1)) + 
    theme(legend.position="none") 
    
    #geom_ribbon(aes(x=id, ymin=AgeSex, ymax=`Age+Sex+MedicalHistory`), fill="red", alpha=0.2)
#geom_violin(size=0.1)
overview

In [ ]:
library(gt)
plot_name = "Figure3a_overview"
overview %>% ggsave(filename=glue("outputs/{plot_name}.pdf"), device="pdf", width=plot_width, height=plot_height, dpi=plot_res)

In [ ]:
temp %>% write_feather("outputs/cindexdeltas.feather")
temp %>% write_csv("outputs/cindexdeltas.csv")

In [ ]:
temp %>% ungroup() %>% group_by(.lower>0) %>% tally()

In [ ]:
1800/1883

In [ ]:
temp %>% ungroup() %>% filter(.lower>0) %>% median_qi(delta, .width = c(.25)) %>% select(.lower, delta, .upper)

In [ ]:
base_size = 8
title_size = 10
facet_size = 9
geom_text_size=3
theme_set(theme_classic(base_size = base_size) + 
          theme(strip.background = element_blank(), plot.title=element_text(size=title_size, hjust=0), 
                strip.text.x = element_text(size = facet_size),axis.title=element_text(size=10), axis.text=element_text(size=8, color="black"),
                legend.position="bottom", axis.line = element_line(size = 0.2), axis.ticks=element_line(size=0.2), panel.grid.major=element_line()))

## Performance against CVD Scores

In [ ]:
SCORE2: Myocardial infarction, Stroke, hypertensive heart disease, Ischemic heart disease, heart failure, arrhythmias, cardiovascular death + cardiac arrest
ASCVD: 
QRISK3:  Ischemic heart disease, Myocardial infarction, TIA, stroke

AF: some scores ESC


In [ ]:
endpoint_selection

In [ ]:
plot_width = 8.25; plot_height=2; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)

plot_against_score = function(score1, score2, endpoint_order=c()){
    
    scores_plot = c(score1, score2)#, "AgeSexMedicalHistory"
    
    score_label = glue("{score1} vs. {score2}")
    #print(score_label)

    temp = benchmark_endpoints %>% 
        filter(score %in% scores_plot) %>% 
        filter(endpoint %in% endpoint_selection) %>%
        mutate(score = factor(score, levels=scores_plot)) %>%
        mutate(endpoint = factor(endpoint, levels=sort(endpoint_selection))) %>%
        group_by(endpoint, score, phecode_string, phecode_category) %>%
        ungroup() %>%
        pivot_wider(names_from=score, values_from=cindex) %>% 
        mutate(id = row_number()) %>%
        mutate(delta = !!sym(score2)-!!sym(score1)) %>%
        mutate(pos = case_when(delta>=0 ~ "pos", delta < 0 ~"neg")) %>%
        mutate(endpoint = fct_reorder(endpoint, delta)) %>%
        mutate(highlight = case_when(endpoint %in% endpoint_selection ~ "YES", TRUE ~ "NO")) %>%# %>% filter(endpoint %in% endpoint_sample)
        mutate(phecode_category = factor(phecode_category, levels=categories_sorted)) %>%
        filter(endpoint %in% endpoint_selection) #%>% mutate(endpoint=factor(endpoint, levels=endpoint_order_diff))
    
    temp_abs = temp %>% group_by(endpoint) %>% summarise(delta=median(delta), m_score2=median(!!sym(score2)), m_score1=median(!!sym(score1))) %>% ungroup() 
    #print(levels(temp_abs$endpoint))
    
    temp_abs_segment = temp_abs %>% rowwise() %>% mutate(min_cindex = min(m_score1, m_score2), max_cindex=max(m_score1, m_score2)) %>% ungroup()# %>% filter(abs(min_cindex-max_cindex)>0.02) 
    
    endpoint_order = (temp %>% group_by(endpoint) %>% summarise(delta=median(delta)) %>% arrange(delta))$endpoint
    print(endpoint_order)
    
    if (length(endpoint_order)>0){
        temp = temp %>% filter(endpoint %in% endpoint_order) %>% mutate(endpoint=factor(endpoint, levels=endpoint_order))
        temp_abs = temp_abs %>% filter(endpoint %in% endpoint_order) %>% mutate(endpoint=factor(endpoint, levels=endpoint_order))
        temp_abs_segment = temp_abs_segment %>% filter(endpoint %in% endpoint_order) %>% mutate(endpoint=factor(endpoint, levels=endpoint_order))
        }
    #print(temp_abs_segment)
    
    abs = ggplot(temp_abs) + 
        #geom_violin(aes(x=fct_rev(endpoint), y=delta), size=0.5) +
        labs(y="Concordance Index")+

        #geom_segment(data=temp_abs_segment, mapping=aes(x=endpoint, xend=endpoint, y=min_cindex+0.01, yend=max_cindex-0.01), alpha=0.4)+#, arrow = arrow(length = unit(0.01, "npc")), arrow.fill="black")+#+

    
        geom_point(aes(x=fct_rev(endpoint), y=m_score1), size=1, color="black", alpha=0.7)+
        #geom_point(aes(x=fct_rev(endpoint), y=m_asm), size=1, color="#023768", alpha=0.7)+
        geom_point(aes(x=fct_rev(endpoint), y=m_score2), size=1.5, color="firebrick", alpha=0.7)+
        geom_segment(data=temp_abs %>% filter(abs(delta)>0.02) %>% mutate(endpoint=factor(endpoint, levels=endpoint_order)), 
                     aes(x=fct_rev(endpoint), xend=fct_rev(endpoint), y=m_score1+0.01, yend=m_score2-0.01), alpha=0.2, arrow = arrow(length = unit(0.01, "npc")), arrow.fill="black")+#,

        scale_x_discrete(labels=endpoint_map) +

        coord_flip(ylim=c(0.5, 0.9))+
         theme(strip.text = element_text(angle=270), axis.title.y=element_blank()) + 
        theme(legend.position="none")

        #geom_ribbon(aes(x=id, ymin=AgeSex, ymax=`Age+Sex+MedicalHistory`), fill="red", alpha=0.2)
    #geom_violin(size=0.1)
    rel = ggplot(temp) + 
        #geom_violin(aes(x=fct_rev(endpoint), y=delta), size=0.5) +
        labs(y="Difference in Concordance Index")+
        geom_hline(yintercept=0, size=0.25, alpha=0.5, linetype="22") + 
        stat_pointinterval(aes(x=fct_rev(endpoint), y=delta), size=0.5, alpha=0.7)+

        theme(axis.title.y=element_blank(),
            axis.text.y=element_blank(),
           axis.ticks.y=element_blank()) + 
        coord_flip(ylim=c(-0.01, 0.23))+
         theme(strip.text = element_text(angle=270)) + 
        theme(legend.position="none")

        #geom_ribbon(aes(x=id, ymin=AgeSex, ymax=`Age+Sex+MedicalHistory`), fill="red", alpha=0.2)
    #geom_violin(size=0.1) 
    return(abs|rel)
    }

In [ ]:
library(ggdist)

In [ ]:
base_size = 8
title_size = 10
facet_size = 9
geom_text_size=3
theme_set(theme_classic(base_size = base_size) + 
          theme(strip.background = element_blank(), plot.title=element_text(size=title_size, hjust=0), 
                strip.text.x = element_text(size = facet_size),axis.title=element_text(size=10), axis.text=element_text(size=8, color="black"),
                legend.position="bottom", axis.line = element_line(size = 0.2), axis.ticks=element_line(size=0.2), panel.grid.major=element_line()))

In [ ]:
endpoint_order=c()
length(endpoint_order)

In [ ]:
sort(endpoint_selection)

In [ ]:
plot_width = 8.25; plot_height=3.25; plot_res = 320
options(repr.plot.width = plot_width, repr.plot.height = plot_height, repr.plot.res=plot_res)

fig3b = plot_against_score("Age+Sex", "Age+Sex+MedicalHistory")
fig3b

In [ ]:
library(gt)
plot_name = "Figure3b_performances"
fig3b %>% ggsave(filename=glue("outputs/{plot_name}.pdf"), device=cairo_pdf, width=plot_width, height=plot_height, dpi=plot_res)

In [ ]:
library(ggdist)

In [ ]:
table_2 = benchmark_endpoints %>% 
    filter(score %in% scores_plot) %>% 
    mutate(score = factor(score, levels=scores_plot)) %>%
    mutate(endpoint = factor(endpoint, levels=endpoints_sorted)) %>%
    group_by(endpoint, score, phecode_string, phecode_category) %>%
    pivot_wider(names_from=score, values_from=cindex) %>% 
    mutate(id = row_number()) %>%
    mutate(delta = `Age+Sex+MedicalHistory`-`Age+Sex`) %>%
    #select(endpoint, iteration, phecode_string, phecode_category, sex, `Age+Sex`, `Age+Sex+MedicalHistory`, delta) %>%
    pivot_longer(all_of(c("Age+Sex", "Age+Sex+MedicalHistory", "delta")), names_to="type", values_to="cindex") %>%
    group_by(endpoint, phecode_string, phecode_category, type) %>%
    median_qi(cindex) %>%
    #ungroup() %>%
    mutate(agg = glue("{round(cindex, 3)} ({round(.lower, 3)}, {round(.upper, 3)})")) %>%
    ungroup() %>% select(endpoint, phecode_string, phecode_category, type, agg) %>%
    pivot_wider(names_from=type, values_from=agg)
    #mutate(pos = case_when(delta>=0 ~ "pos", delta < 0 ~"neg")) %>%
    #mutate(endpoint = fct_reorder(endpoint, delta))# %>% filter(endpoint %in% endpoint_sample)

In [ ]:
table_2 %>% 
    select(all_of(c("endpoint", "phecode_string", "Age+Sex", 'Age+Sex+MedicalHistory', "delta"))) %>%
    mutate(endpoint = factor(endpoint, levels = endpoints_md$endpoint)) %>% 
    filter(endpoint %in% endpoint_selection) %>%
    arrange(endpoint)

In [ ]:
options(pillar.print_max = Inf)
table_2 %>% 
    select(all_of(c("phecode_category", "endpoint", "phecode_string", "Age+Sex", 'Age+Sex+MedicalHistory', "delta"))) %>%
    mutate(endpoint = factor(endpoint, levels = endpoints_md$endpoint)) %>% 
    #filter(endpoint %in% endpoint_selection) %>%
    arrange(endpoint) %>% 
    write_csv("outputs/SupplTable5_DiscriminativePerformanceAll.csv")

In [ ]:
table_2 %>% filter(endpoint %in% endpoint_selection) %>% arrange(as.character(endpoint)) %>% arrange(delta)